## **1\. Find the Most Recent Order for Each Customer**

In [24]:
SELECT OrderID, CustomerID, OrderDate
FROM Sales.Orders O
WHERE OrderDate = (
    SELECT MAX(OrderDate) 
    FROM Sales.Orders 
    WHERE CustomerID = O.CustomerID
);


(785 rows affected)

Total execution time: 00:00:00.094

OrderID,CustomerID,OrderDate
69549,905,2016-03-31
69588,905,2016-03-31
70159,66,2016-04-11
70223,128,2016-04-12
70240,128,2016-04-12
70298,15,2016-04-13
70300,15,2016-04-13
70324,131,2016-04-13
70330,130,2016-04-13
70352,131,2016-04-13


## **2\. Find Customers Who Have Never Placed an Order**

In [25]:
SELECT CustomerID, CustomerName
FROM Sales.Customers C
WHERE NOT EXISTS (
    SELECT 1 
    FROM Sales.Orders O 
    WHERE O.CustomerID = C.CustomerID
);

(0 rows affected)

Total execution time: 00:00:00.022

CustomerID,CustomerName


## **3\. Find Customers Who Placed Orders Only in 2023**

In [26]:
SELECT CustomerID, CustomerName
FROM Sales.Customers C
WHERE CustomerID IN (
    SELECT CustomerID FROM Sales.Orders WHERE YEAR(OrderDate) = 2023
)
AND CustomerID NOT IN (
    SELECT CustomerID FROM Sales.Orders WHERE YEAR(OrderDate) <> 2023
);


(0 rows affected)

Total execution time: 00:00:00.026

CustomerID,CustomerName


## **4\. Find Orders with the Highest Total Value**

In [27]:
SELECT OrderID, CustomerID, 
       (SELECT SUM(Quantity * UnitPrice) 
        FROM Sales.OrderLines 
        WHERE OrderID = O.OrderID) AS TotalValue
FROM Sales.Orders O
WHERE (SELECT SUM(Quantity * UnitPrice) 
       FROM Sales.OrderLines 
       WHERE OrderID = O.OrderID) = 
      (SELECT MAX(TotalAmount) 
       FROM (SELECT OrderID, SUM(Quantity * UnitPrice) AS TotalAmount 
             FROM Sales.OrderLines 
             GROUP BY OrderID) AS OrderSums);


(1 row affected)

Total execution time: 00:00:00.186

OrderID,CustomerID,TotalValue
30269,834,32026.00


## **5\. Find Products That Have Never Been Ordered**

In [29]:
SELECT StockItemID, StockItemName
FROM Warehouse.StockItems P
WHERE NOT EXISTS (
    SELECT 1 
    FROM Sales.OrderLines OL 
    WHERE OL.StockItemID = P.StockItemID
);


(0 rows affected)

Total execution time: 00:00:00.030

StockItemID,StockItemName


## **6\. Find the Maximum Orders for Each Salesperson Using a Derived Table**

In [30]:
SELECT E.PersonID, E.FullName, MaxOrders.Max_Orders
FROM Application.People E
INNER JOIN (
    SELECT SalespersonPersonID, COUNT(OrderID) AS Max_Orders 
    FROM Sales.Orders 
    GROUP BY SalespersonPersonID
) AS MaxOrders
ON E.PersonID = MaxOrders.SalespersonPersonID;


(10 rows affected)

Total execution time: 00:00:00.028

PersonID,FullName,Max_Orders
2,Kayla Woodcock,7474
3,Hudson Onslow,7281
6,Sophia Hinton,7349
7,Amy Trefl,7276
8,Anthony Grosse,7257
13,Hudson Hollinworth,7400
14,Lily Code,7268
15,Taj Shand,7371
16,Archer Lamble,7532
20,Jack Potter,7387


## **7\. Rank Salespeople by Total Sales**

In [31]:
WITH SalesRanking AS (
    SELECT O.SalespersonPersonID, E.FullName,
           SUM(OL.Quantity * OL.UnitPrice) AS TotalSales
    FROM Sales.Orders O
    JOIN Application.People E ON O.SalespersonPersonID = E.PersonID
    JOIN Sales.OrderLines OL ON O.OrderID = OL.OrderID
    GROUP BY O.SalespersonPersonID, E.FullName
)
SELECT SalespersonPersonID, FullName, TotalSales,
       ROW_NUMBER() OVER (ORDER BY TotalSales DESC) AS SalesRank
FROM SalesRanking;


(10 rows affected)

Total execution time: 00:00:00.127

SalespersonPersonID,FullName,TotalSales,SalesRank
16,Archer Lamble,18551146.95,1
2,Kayla Woodcock,18107095.00,2
3,Hudson Onslow,17815605.10,3
15,Taj Shand,17812364.60,4
6,Sophia Hinton,17768199.25,5
13,Hudson Hollinworth,17716354.25,6
20,Jack Potter,17621145.20,7
14,Lily Code,17612639.80,8
7,Amy Trefl,17329344.05,9
8,Anthony Grosse,17300382.20,10


## **8\. Find Customers with a Running Total of Orders**

In [32]:
WITH CustomerOrderTotals AS (
    SELECT CustomerID, OrderDate, 
           COUNT(OrderID) AS OrderCount,
           SUM(COUNT(OrderID)) OVER (PARTITION BY CustomerID ORDER BY OrderDate) AS RunningTotal
    FROM Sales.Orders
    GROUP BY CustomerID, OrderDate
)
SELECT * FROM CustomerOrderTotals;


(62513 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.419

CustomerID,OrderDate,OrderCount,RunningTotal
6,2013-01-05,1,1
6,2013-01-14,1,2
6,2013-02-21,1,3
6,2013-03-15,1,4
6,2013-03-21,1,5
6,2013-04-05,1,6
6,2013-04-26,1,7
6,2013-05-09,1,8
6,2013-05-15,2,10
6,2013-06-11,1,11


## **9\. Find the Two Most Expensive Products for Each Supplier**

In [33]:
SELECT S.SupplierID, S.SupplierName, P.StockItemID, P.StockItemName, P.UnitPrice
FROM Purchasing.Suppliers S
CROSS APPLY (
    SELECT TOP 2 StockItemID, StockItemName, UnitPrice 
    FROM Warehouse.StockItems 
    WHERE SupplierID = S.SupplierID 
    ORDER BY UnitPrice DESC
) AS P;


(14 rows affected)

Total execution time: 00:00:00.030

SupplierID,SupplierName,StockItemID,StockItemName,UnitPrice
1,A Datum Corporation,221,Novelty chilli chocolates 500g,14.50
1,A Datum Corporation,222,Chocolate beetles 250g,8.55
2,"Contoso, Ltd.",152,Pack of 12 action figures (female),16.00
2,"Contoso, Ltd.",151,Pack of 12 action figures (male),16.00
4,"Fabrikam, Inc.",102,Alien officer hoodie (Black) XL,35.00
4,"Fabrikam, Inc.",103,Alien officer hoodie (Black) XXL,35.00
5,Graphic Design Institute,18,DBA joke mug - daaaaaa-ta (White),13.00
5,Graphic Design Institute,17,DBA joke mug - mind if I join you? (Black),13.00
7,"Litware, Inc.",215,Air cushion machine (Blue),1899.00
7,"Litware, Inc.",174,Bubblewrap dispenser (Black) 1.5m,240.00


## **10\. Find the Management Chain for an Employee**

In [34]:
WITH MonthlySales AS (
    SELECT 
        CustomerID,
        FORMAT(OrderDate, 'yyyy-MM') AS OrderMonth, -- Extract year-month format
        SUM(Quantity * UnitPrice) AS TotalSales
    FROM Sales.Orders O
    JOIN Sales.OrderLines OL ON O.OrderID = OL.OrderID
    GROUP BY CustomerID, FORMAT(OrderDate, 'yyyy-MM')
)
SELECT * FROM MonthlySales
ORDER BY CustomerID, OrderMonth;


(23871 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.508

CustomerID,OrderMonth,TotalSales
1,2013-03,10338.75
1,2013-04,16747.95
1,2013-05,3060.00
1,2013-06,12210.50
1,2013-07,6092.80
1,2013-08,2642.60
1,2013-09,10916.00
1,2013-10,6848.20
1,2013-11,2320.00
1,2013-12,8968.00
